In [1]:
%matplotlib inline
import random
import torch
from torch.utils import data
import numpy as np


In [2]:
#人造数据集
def synthetic_data(w, b, num_examples) :
    '''生成 y = Xw + b + 噪声'''
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape(-1,1)

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

In [6]:
#调用框架中现有API来读取数据
def load_array(data_arrays, batch_size, is_train = True) :
    '''构造一个pytorch数据迭代器'''
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle = is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)
next(iter(data_iter))

[tensor([[-1.0837, -0.0122],
         [ 1.6639, -0.6699],
         [ 0.6325, -0.9817],
         [-0.6228, -0.0683],
         [ 0.0127, -0.5111],
         [ 0.6312, -0.5729],
         [-0.0394,  1.6049],
         [ 0.6088, -0.8339],
         [-0.9185, -1.0308],
         [ 1.7483,  0.1244]]),
 tensor([[ 2.0720],
         [ 9.8086],
         [ 8.8224],
         [ 3.1920],
         [ 5.9753],
         [ 7.3992],
         [-1.3312],
         [ 8.2436],
         [ 5.8501],
         [ 7.2878]])]

In [6]:
#使用框架的预定义好的层
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))

Parameter containing:
tensor([[-0.0029, -0.5976]], requires_grad=True)

In [8]:
#初始化模型参数
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

In [9]:
#均方误差
loss = nn.MSELoss()


In [10]:
#实例化SGD
trainer = torch.optim.SGD(net.parameters(), lr = 0.03)

In [11]:
#训练过程
num_epochs = 3
for epoch in range(num_epochs) :
    for X, y in data_iter:
        l = loss(net(X), y)
        trainer.zero_grad()
        l.backward()
        trainer.step()

    l = loss(net(features), labels)
    print(f"epoch {epoch + 1}, loss {l : f}")

epoch 1, loss  0.000210
epoch 2, loss  0.000105
epoch 3, loss  0.000105
